In [4]:
from datasets import load_dataset
import numpy as np

In [20]:
ds = load_dataset("afmck/text8") # https://huggingface.co/datasets/afmck/text8

## The Skip-gram Model
the objective of the Skip-gram model is to maximize the average log probability

$$
\frac{1}{T} \sum_{t=1}^{T} \sum_{\substack{-c \le j \le c,j \ne 0}} \log p(w_{t+j} \mid w_t)
$$
- $c$ is the size of the training context (which can be a function of the center word $w_{t}$)

### Softmax

The basic Skip-gram formulation defines $p(w_{t+j} \mid w_t)$ using the softmax function:

$$
p(w_O \mid w_I) = \frac{\exp\left({v'_{w_O}}^{\top} v_{w_I}\right)}{\sum_{w=1}^{W} \exp\left({v'_w}^{\top} v_{w_I}\right)}
$$
- $v_{w}$ and $v'_{w}$ are the “input” and “output” vector representations of $w$, and $W$ is the number of words in the vocabulary.<br>
- This formulation is impractical because the cost of computing $\nabla \log p(w_O \mid w_I)$ is proportional to $W$, which is often very large ($10^5$–$10^7$ terms).

### Hierarchical Softmax

Then the hierarchical softmax defines $p(w_O \mid w_I)$ as follows:

$$
p(w \mid w_I) = \prod_{j=1}^{L(w)-1} \sigma \left( [\!\left[ n(w, j+1) = ch(n(w, j)) \right]\!\right] \cdot {v'}_{n(w,j)}^{\top} v_{w_I} )
$$

- $\sigma(x) = {1} / {(1 + \exp(-x))}$. It can be verified that $\sum_{w=1}^{W} p(w \mid w_I) = 1$
- let $ch(n)$ be an arbitrary fixed child of $n$
- let $ \left[\!\left[ x \right]\!\right] $ be 1 if $x$ is true and -1 otherwise

### Negative Sampling

An alternative to the hierarchical softmax is Noise Contrastive Estimation (NCE)

We define Negative sampling (NEG) by the objective 

$$
\log \sigma\left({v'}_{w_O}^{\top} v_{w_I}\right) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma\left(-{v'}_{w_i}^{\top} v_{w_I}\right) \right]
$$

which is used to replace every $\log P (w_O | w_I )$ term in the Skip-gram objective

- values of k in the range 5–20 are useful for small training datasets
- for large datasets the k can be as small as 2–5

> The main difference between the Negative sampling and NCE is that NCE needs both samples and the numerical probabilities of the noise distribution, while Negative sampling uses only samples

#### Noise Distribution

Both NCE and NEG have the noise distribution $P_n(w)$ as a free parameter

- unigram distribution $U(w)$ raised to the 3/4rd power (i.e., $U(w)^{3/4}/Z$) outperformed significantly the unigram and the uniform distributions

### Subsampling of Frequent Words

To counter the imbalance between the rare and frequent words, we used a simple subsampling approach: each word $w_i$ in the training set is discarded with probability computed by the formula

$$
P(w_i) = 1 - \sqrt{ \frac{t}{f(w_i)} }
$$

- $f(w_i)$ is the frequency of word $w_i$
- $t$ is a chosen threshold, typically around $10^{-5}$

We chose this subsampling formula because it aggressively subsamples words whose frequency is greater than $t$ while preserving the ranking of the frequencies